In [4]:
import pandas as pd
df = pd.read_csv(r"C:\Users\aniruddh.singh\Documents\Project_1\data\landing\sms-call-internet-mi-2013-11-01.csv")

In [5]:
print(df.head())
print(df.shape)
print(df.columns)
print(df.dtypes)

              datetime  CellID  countrycode   smsin  smsout  callin  callout  \
0  2013-11-01 00:00:00       1            0  0.3521     NaN     NaN   0.0273   
1  2013-11-01 00:00:00       1           33     NaN     NaN     NaN      NaN   
2  2013-11-01 00:00:00       1           39  1.7322  1.1047  0.5919   0.4020   
3  2013-11-01 00:00:00       2            0  0.3581     NaN     NaN   0.0273   
4  2013-11-01 00:00:00       2           33     NaN     NaN     NaN      NaN   

   internet  
0       NaN  
1    0.0261  
2   57.7729  
3       NaN  
4    0.0274  
(1891928, 8)
Index(['datetime', 'CellID', 'countrycode', 'smsin', 'smsout', 'callin',
       'callout', 'internet'],
      dtype='str')
datetime           str
CellID           int64
countrycode      int64
smsin          float64
smsout         float64
callin         float64
callout        float64
internet       float64
dtype: object


In [6]:
mapping = {'datetime':'timestamp',
                     'CellID':'grid_id',
'countrycode':'country_code',
'smsin':'sms_in',
'smsout':'sms_out',
'callin':'call_in',
'callout':'call_out',
'internet':'internet_activity'}

df = df.rename(columns=mapping)

In [7]:
df['sms_in']=df['sms_in'].fillna(0)
df['sms_out']=df['sms_out'].fillna(0)
df['call_in']=df['call_in'].fillna(0)
df['call_out']=df['call_out'].fillna(0)
df['internet_activity']=df['internet_activity'].fillna(0)

In [8]:
#Derived Activity Measures
df['total_sms'] = df['sms_in'] + df['sms_out']
df['total_calls'] = df['call_in'] + df['call_out']
df['total_activity'] = df['total_sms'] + df['total_calls']+df['internet_activity']

In [9]:
print(df.columns)

Index(['timestamp', 'grid_id', 'country_code', 'sms_in', 'sms_out', 'call_in',
       'call_out', 'internet_activity', 'total_sms', 'total_calls',
       'total_activity'],
      dtype='str')


In [10]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [11]:
print("Number of unique datetime stamps:",df['timestamp'].nunique())
unique_hours = df['timestamp'].dt.hour.unique()
valid = True
for i in range(len(unique_hours)):
    print(unique_hours[i],unique_hours[i-1])

Number of unique datetime stamps: 24
0 23
1 0
2 1
3 2
4 3
5 4
6 5
7 6
8 7
9 8
10 9
11 10
12 11
13 12
14 13
15 14
16 15
17 16
18 17
19 18
20 19
21 20
22 21
23 22


In [12]:
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

print(df[['timestamp','date','hour','day_of_week']])


                  timestamp        date  hour  day_of_week
0       2013-11-01 00:00:00  2013-11-01     0            4
1       2013-11-01 00:00:00  2013-11-01     0            4
2       2013-11-01 00:00:00  2013-11-01     0            4
3       2013-11-01 00:00:00  2013-11-01     0            4
4       2013-11-01 00:00:00  2013-11-01     0            4
...                     ...         ...   ...          ...
1891923 2013-11-01 23:00:00  2013-11-01    23            4
1891924 2013-11-01 23:00:00  2013-11-01    23            4
1891925 2013-11-01 23:00:00  2013-11-01    23            4
1891926 2013-11-01 23:00:00  2013-11-01    23            4
1891927 2013-11-01 23:00:00  2013-11-01    23            4

[1891928 rows x 4 columns]


In [13]:
print("Number of missing grid Ids:",df['grid_id'].isna().sum())
print("Number of missing timestamps:",df['timestamp'].isna().sum())
print("Number of records with blank as SMS count:",df['total_sms'].isna().sum())
print("Number of records with blank as call count:",df['total_calls'].isna().sum())
print("Number of records with blank as internet usage:",df['internet_activity'].isna().sum())



Number of missing grid Ids: 0
Number of missing timestamps: 0
Number of records with blank as SMS count: 0
Number of records with blank as call count: 0
Number of records with blank as internet usage: 0


In [14]:
print("Any exact duplicates?\n",df.duplicated().any())

Any exact duplicates?
 False


In [15]:
if ((df['total_sms'] < 0) |
    (df['total_calls'] < 0) |
    (df['internet_activity'] < 0)).any():
    print("Negative values found")
else:
    print("All values are positive")

All values are positive


In [ ]:
df.groupby(['grid_id', 'hour'])['country_code'].count()

grid_id  hour
1        0       3
         1       2
         2       3
         3       2
         4       2
                ..
10000    19      7
         20      2
         21      7
         22      8
         23      6
Name: country_code, Length: 240000, dtype: int64

In [20]:
df['raw_grain']=(df['timestamp'].astype(str)+'_'+df['grid_id'].astype(str)+'_'+df['country_code'].astype(str))

In [21]:
df.duplicated(
    subset=['timestamp', 'grid_id', 'country_code']
).sum()

np.int64(0)

In [22]:
print("Number of unique grid IDs:",df['grid_id'].nunique())
print("Max timestamp:",df['timestamp'].max())
print("Min timestamp:",df['timestamp'].min())
print("Cadence:",unique_hours[1]-unique_hours[0])
print("Counrty code categories:",df['country_code'].unique())


Number of unique grid IDs: 10000
Max timestamp: 2013-11-01 23:00:00
Min timestamp: 2013-11-01 00:00:00
Cadence: 1
Counrty code categories: [    0    33    39    64    46    32    40    34    49    81     7   593
   972    20    44    41     1    86    31    53    63   880    92   212
   221    43   351    48    51   380    55    58    62    45   503   353
    30   233    47    57   242    90   421   420   973   385  1780    84
   373   372    52   216    54    36    82   213    98 88239   595    61
   370   356   354    93   591   856   258   352   852  1514    94   357
   358  1787   359  1829    60   598   381   244  1671   974   371   389
   971   225   234   966   355   382    65   507   375    56    66   238
  1519   218  1613  8817   226 18098   241   386 18099   223   994 18093
    91   965  1416  1579  1778  1705   886   237   976    95   675   230
  1849  1647   265   504   243   968  7705  1438  1587 18096  7778   261
   255  1214   506    27   251 18094   263  1905  7702  77

In [33]:
max_activity = df['total_activity'].max()
busiest_hour_window = df.loc[df['total_activity'] == max_activity, ['timestamp','total_activity']]
print("Busiest hourly window:\n", busiest_hour_window.iloc[0])
busiest_grid = df.loc[df['total_activity'] == max_activity, ['grid_id','total_activity']]
print("Busiest grid:\n", busiest_grid.iloc[0])
print("Nulls per column\n", df.isnull().sum())

Busiest hourly window:
 timestamp         2013-11-01 17:00:00
total_activity             31383.9979
Name: 1326772, dtype: object
Busiest grid:
 grid_id            5161.0000
total_activity    31383.9979
Name: 1326772, dtype: float64
Nulls per column
 timestamp            0
grid_id              0
country_code         0
sms_in               0
sms_out              0
call_in              0
call_out             0
internet_activity    0
total_sms            0
total_calls          0
total_activity       0
date                 0
hour                 0
day_of_week          0
raw_grain            0
dtype: int64
